In [1]:
import pandas as pd
import numpy as np

In [2]:
df_SpamDataset = pd.read_csv("../DataPreProcessing/final_balanced_dataset.csv", encoding="latin1")

In [3]:
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report


In [4]:
df_SpamDataset.columns

Index(['label', 'text'], dtype='object')

In [5]:
df_SpamDataset.head()

,label,text
0,Spam,encouragement sexual functioning http wilson j...
1,Ham,citibank just turned us down on a qescapenumbe...
2,Spam,ÃÂÃÂªÃÂÃÂªÃÂÃÂªÃÂÃÂªÃÂÃÂªÃÂÃ...
3,Ham,im original message dinh huy sent monday novem...
4,Spam,the internet sector is hot again search for as...


In [6]:
df_SpamDataset.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21320 entries, 0 to 21319
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   21320 non-null  object
 1   text    21320 non-null  object
dtypes: object(2)
memory usage: 333.3+ KB


In [7]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)   # remove links
    text = re.sub(r'\d+', '', text)       # remove numbers
    text = re.sub(r'[^a-zA-Z ]', '', text)  # remove special characters
    text = re.sub(r'\s+', ' ', text)      # remove extra spaces
    return text.strip()



In [8]:
df_SpamDataset["Text"] = df_SpamDataset["text"].astype(str).apply(clean_text)
df_SpamDataset.head()


,label,text,Text
0,Spam,encouragement sexual functioning http wilson j...,encouragement sexual functioning http wilson j...
1,Ham,citibank just turned us down on a qescapenumbe...,citibank just turned us down on a qescapenumbe...
2,Spam,ÃÂÃÂªÃÂÃÂªÃÂÃÂªÃÂÃÂªÃÂÃÂªÃÂÃ...,z h http www angelfire com folk escapelong htt...
3,Ham,im original message dinh huy sent monday novem...,im original message dinh huy sent monday novem...
4,Spam,the internet sector is hot again search for as...,the internet sector is hot again search for as...


In [9]:
df_SpamDataset.rename(columns={
    "label": "Spam"
}, inplace=True)

In [10]:
df_SpamDataset

,Spam,text,Text
0,Spam,encouragement sexual functioning http wilson j...,encouragement sexual functioning http wilson j...
1,Ham,citibank just turned us down on a qescapenumbe...,citibank just turned us down on a qescapenumbe...
2,Spam,ÃÂÃÂªÃÂÃÂªÃÂÃÂªÃÂÃÂªÃÂÃÂªÃÂÃ...,z h http www angelfire com folk escapelong htt...
3,Ham,im original message dinh huy sent monday novem...,im original message dinh huy sent monday novem...
4,Spam,the internet sector is hot again search for as...,the internet sector is hot again search for as...
...,...,...,...
21315,Spam,= ? utf - 8 ? q ? ? =\r\ngenuine replicas wris...,utf q genuine replicas wrist watchesi have thi...
21316,Spam,beefsteak arboretum mountainous roughshod alle...,beefsteak arboretum mountainous roughshod alle...
21317,Ham,and the hype begins\r\nrod williams\r\n07 / 13...,and the hype beginsrod williams amto phil lowr...
21318,Ham,"mike ,\r\nas a vision statement this looks lik...",mike as a vision statement this looks like an ...


In [11]:
X = df_SpamDataset["Text"]
y = df_SpamDataset["Spam"]


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [13]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.9,
    min_df=2,
    ngram_range=(1,2)
)




In [14]:
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [15]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_vec, y_train)


LogisticRegression(class_weight='balanced', max_iter=1000)

In [16]:
y_pred = model.predict(X_test_vec)


In [17]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.9598968105065666

Classification Report:

              precision    recall  f1-score   support

         Ham       0.96      0.95      0.96      2032
        Spam       0.96      0.97      0.96      2232

    accuracy                           0.96      4264
   macro avg       0.96      0.96      0.96      4264
weighted avg       0.96      0.96      0.96      4264



In [18]:
import pickle

pickle.dump(model, open("spam_model.pkl", "wb"))
pickle.dump(vectorizer, open("spam_vectorizer.pkl", "wb"))


In [19]:
df_SpamDataset["Spam"].value_counts()


Spam
Spam    11160
Ham     10160
Name: count, dtype: int64

In [20]:
import pickle

# Save model
with open("spam_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("spam_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)